In [ ]:
import json
from bs4 import BeautifulSoup, Tag
import requests
import re
from display_helpers import pretty_print_page
from concurrent.futures import ThreadPoolExecutor
from data_helpers import load_cache, save_cache, merge_article_into_cache
import time
from datetime import datetime, timezone
import os
from urllib.parse import urlparse, unquote
from typing import Union, List
import sqlite3
from random import sample, choices




In [ ]:
#-----------------------------
# (try to) convert simple wiki page url to its classic wiki page url 
# counterpart
# Goal -> have a normal wikipedia page for each simple wikipedia page 
#         already stored

def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url


#-----------------------------
# And conversely....

def normal2simplewiki_url(normal_url):
    page_url_segment = normal_url.split("/")[-1]
    simplewiki_base = "https://simple.wikipedia.org/wiki/"
    simplewiki_url = simplewiki_base + page_url_segment
    return simplewiki_url


# Normal Wikipedia page scraper

In [ ]:
# ----------------------------------------
# 			  SCRAPING METHOD
# ----------------------------------------

# ---------------- CONFIG ----------------
IGNORE_CLASSES = {
    "sidebar-list", "navbar", "infobox", "toc",
    "thumb", "mw-default-size", "metadata"
}

STOP_SECTIONS = {
    "references", "external links", "see also", "notes", "further reading"
}


# ---------------- HELPERS ----------------
def clean_paragraph(el: Tag) -> str:
    """Clean paragraph text, preserving math as LaTeX."""

    # Remove citation markers
    for sup in el.find_all("sup"):
        sup.decompose()

    # Preserve math
    for math in el.find_all("math"):
        latex = math.get("alttext") or math.get_text(strip=True)
        latex = latex.strip()

        is_block = el.get_text(strip=True) == math.get_text(strip=True)
        math.replace_with(
            f"\n$$\n{latex}\n$$\n" if is_block else f"${latex}$"
        )

    # Lists
    if el.name in {"ul", "ol"}:
        lines = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            txt = clean_paragraph(li)
            if txt:
                lines.append(f"- {txt}" if el.name == "ul" else f"{i}) {txt}")
        return "\n".join(lines)

    # Text cleanup
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


def is_ignored(el: Tag) -> bool:
    """Ignore elements inside navboxes, infoboxes, thumbnails, TOC."""
    for parent in el.parents:
        classes = parent.get("class", [])
        if any(cls in IGNORE_CLASSES for cls in classes):
            return True
    return False



# ---------------- MAIN SCRAPER ----------------
def scrape_normal_wiki(url: str) -> dict:
    headers = {"User-Agent": "ReverseMentorBot/0.1"}
    
    try:
         res = requests.get(url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch URL: {url}") from e


    soup = BeautifulSoup(res.text, "html.parser")


    content = soup.find("div", id="mw-content-text")
    
    if content is None:
        raise ValueError(f"Content div not found for {url}")

    sections = []
    intro = None
    current = None

    # Traverse in DOM order
    for el in content.find_all(
        ["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"],
        recursive=True
    ):
        if is_ignored(el):
            continue

        # ---------- HEADINGS ----------
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in STOP_SECTIONS:
                break

            current = {"heading": heading, "paragraphs": []}
            sections.append(current)
            continue

        # ---------- CONTENT ----------
        text = clean_paragraph(el)
        if not text:
            continue

        if current is None:
            if intro is None:
                intro = {"heading": "Introduction", "paragraphs": []}
                sections.insert(0, intro)
            intro["paragraphs"].append(text)
        else:
            current["paragraphs"].append(text)

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    return {
        "url": url,
        "title": title,
        "sections": sections
    }



# Simple wiki page scraper

In [ ]:

# ---------------- HELPERS ----------------

def clean_spaces(text):
    return " ".join(text.split())

def clean_paragraph(el: Tag):
    """
    Clean paragraph text, preserving formulas as LaTeX.
    Handles <p>, <li>, <dd>, <ul>, <ol> elements.
    """

    # Remove citation superscripts
    for sup in el.find_all("sup"):
        sup.decompose()

    # Replace <math> elements with LaTeX
    for math in el.find_all("math"):
        latex = math.get("alttext") or "".join(math.strings).strip()
        math.replace_with(f"${latex.strip()}$")

    # Handle lists
    if el.name in ["ul", "ol"]:
        items = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            li_text = clean_paragraph(li)
            if li_text:
                items.append(f"- {li_text}" if el.name == "ul" else f"{i}) {li_text}")
        return "\n".join(items)

    # Handle list items and description items
    if el.name in ["li", "dd"]:
        parts = []
        for child in el.children:
            if isinstance(child, Tag):
                parts.append(clean_paragraph(child))
            else:
                parts.append(str(child))
        text = " ".join(filter(None, parts))
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s+([.,;:!?])", r"\1", text)
        return text.strip()

    # Default text
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


# --- Recursive content iterator ---
def iter_content_elements(el):
    """
    Yield all relevant content elements in document order.
    Skip navboxes, tables, scripts, styles.
    """
    for child in el.children:
        if not isinstance(child, Tag):
            continue

        if child.name in ["table", "script", "style"]:
            continue

        # Skip sideboxes, navboxes, metadata
        if child.name == "div":
            classes = child.get("class") or []
            if any(c in ["navbox", "vertical-navbox", "metadata", "mbox"] for c in classes):
                continue
            yield from iter_content_elements(child)
            continue
        
        # Skip geo/coordinates spans
        if child.name == "span" and any(c in ["geo", "coordinates"] for c in (child.get("class") or [])):
            continue

		# Recursively yield from spans (other inline containers)
        if child.name == "span":
            yield from iter_content_elements(child)
            continue

        # Yield headings and paragraph-like content
        if child.name in ["p", "ul", "ol", "dd"] + [f"h{i}" for i in range(2, 7)]:
            yield child
        else:
            yield from iter_content_elements(child)





In [ ]:

# ---------------- MAIN SCRAPER ----------------

def scrape_simple_wiki(url):
    """
    Scrapes a Simple Wikipedia page and returns structured article data.
    Handles redirects, missing content, and network errors.
    """
    
    headers = {
        "User-Agent": "ReverseMentorBot/0.1 (https://yourdomain.com/contact)"
    }

    stop_sections = {
        "references",
        "other websites",
        "related pages",
        "further reading",
        "external links",
        "see also",
    }

    # --- Fetch page with network error handling ---
    try:
        res = requests.get(url, headers=headers, timeout=10)
        # Raises for 4xx/5xx responses
        res.raise_for_status()
    except requests.RequestException as e:
        # Network/environment-level failure: connection, timeout, HTTP error, invalid URL
        raise RuntimeError(f"Failed to fetch Simple Wikipedia URL: {url}") from e

    soup = BeautifulSoup(res.text, "html.parser")

    # --- Handle redirects ---
    redirect_div = soup.find("div", class_="redirectMsg")
    if redirect_div and redirect_div.find("a"):
        redirect_url = "https://simple.wikipedia.org" + redirect_div.find("a")["href"]
        return {
            "url": url,
            "title": None,
            "sections": [],
            "redirect": redirect_url,
        }

    # --- Main content ---
    content = soup.find("div", class_="mw-parser-output")
    if content is None:
        # Page layout changed or empty page
        raise ValueError(f"Main content not found for {url}")

    # --- Article title with fallback ---
    title_tag = soup.find("h1", id="firstHeading")
    if title_tag:
        title = title_tag.get_text(strip=True)
    else:
        # fallback: use last segment of URL
        title = url.split("/")[-1].replace("_", " ")

    article_data = {
        "url": url,
        "title": title,
        "sections": [],
        "categories": [],
        "category_urls": [],
    }

    # --- Introduction section ---
    intro_section = {"heading": "Introduction", "paragraphs": []}
    current_section = intro_section

    # --- Walk content recursively ---
    for el in iter_content_elements(content):
        # Headings start new sections
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in stop_sections:
                break
            if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
                article_data["sections"].append(intro_section)
            current_section = {"heading": heading, "paragraphs": []}
            article_data["sections"].append(current_section)
            continue

        # Paragraph-like content
        if el.name in ["p", "ul", "ol", "dd"]:
            text = clean_paragraph(el)
            if text:
                current_section["paragraphs"].append(text)

    # --- Ensure intro is included if it has paragraphs ---
    if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
        article_data["sections"].insert(0, intro_section)

    # --- Categories ---
    catlinks = soup.select("#mw-normal-catlinks ul li a")
    for cat in catlinks:
        article_data["categories"].append(cat.get_text(strip=True))
        href = cat.get("href")
        if href and href.startswith("/wiki/"):
            article_data["category_urls"].append("https://simple.wikipedia.org" + href)

    # --- Ensure at least one section exists ---
    if not article_data["sections"]:
        raise ValueError(f"No sections found for {url}")

    return article_data


In [ ]:


# def get_pages_from_category(category_url: str) -> list[str]:
#     """
#     Placeholder function to extract single page URLs from a category URL.
#     In practice, this would scrape the category page and return links.
#     """
#     # Example logic for demonstration
#     return [f"{category_url}/page{i}" for i in range(1, 4)]


# def scrape_simple(urls: Union[str, List[str]]) -> List[str]:
#     """
#     Accepts:
#         - single wiki page URL
#         - list of wiki page URLs
#         - single category URL
#         - list of category URLs
    
#     Automatically detects category URLs by the '/wiki/Category:' substring
#     and expands them into single page URLs.
    
#     Returns a flat list of single page URLs ready for scraping.
#     """
#     # Ensure we have a list
#     if isinstance(urls, str):
#         urls = [urls]
    
#     final_urls = []
    
#     for url in urls:
#         if "/wiki/Category:" in url:
#             # Treat as category page
#             pages = get_pages_from_category(url)
#             final_urls.extend(pages)
#         else:
#             # Normal wiki page
#             final_urls.append(url)
    
#     # Example scraping logic
#     for u in final_urls:
#         print(f"Scraping {u}...")
#         # Your real scraping code here
    
#     return final_urls


In [ ]:
# -------- Get Wiki pages URLs from a Wiki Category URL --------------
# Used for scraping all pages in a category instead of scraping them individually.
# Works for both simple and normal wiki category pages.

def get_category_pages(category_url):

    headers = {
        "User-Agent": "YourBot/1.0 (https://example.com/contact)"
    }
    
    try:
         res = requests.get(category_url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch category URL: {category_url}") from e
		
    
    soup = BeautifulSoup(res.text, "html.parser")

    # Extract category name from URL
    url_parse = urlparse(category_url)
    path = url_parse.path
    category_name = path.split(":")[-1]

    base = url_parse.scheme + "://" + url_parse.netloc # https://simple.wikipedia.org
    pages = []

    for li in soup.select("#mw-pages li a"):
        href = li.get("href")
        title = li.get_text(strip=True)
        if "Template:" in title:
            continue
        if href and href.startswith("/wiki/"):
            pages.append({
                "title": title,
                "url": base + href
            })
            
    if not pages:
            raise ValueError(f"No pages found in category {category_name} at {category_url}")

    return {
        "categories": category_name,
        "category_urls": category_url,
        "pages": pages
    }

In [ ]:
# test get_category_pages()
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Movie_producers_from_New_York_City')

# gets only the 200 first pages in the category (wiki category page structure - category has multiple pages if more than 200 pages in the category)
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Living_people')
# but works if subsecant pages url is provided:
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/w/index.php?title=Category:Living_people&pagefrom=Abrines+Redondo%2C+Alejandro%0AAlejandro+Abrines+Redondo#mw-pages')

# works also for normal wiki categories:
normal_category_test = get_category_pages(category_url='https://en.wikipedia.org/wiki/Category:Database_models')

len(simple_category_test['pages']), simple_category_test['pages']

(200,
 [{'title': 'Alejandro Abrines Redondo',
   'url': 'https://simple.wikipedia.org/wiki/Alejandro_Abrines_Redondo'},
  {'title': 'Anne-Ségolène Abscheidt',
   'url': 'https://simple.wikipedia.org/wiki/Anne-S%C3%A9gol%C3%A8ne_Abscheidt'},
  {'title': 'Mohammad Abshak',
   'url': 'https://simple.wikipedia.org/wiki/Mohammad_Abshak'},
  {'title': 'Mehdi Abtahi',
   'url': 'https://simple.wikipedia.org/wiki/Mehdi_Abtahi'},
  {'title': 'Najmeh Abtin',
   'url': 'https://simple.wikipedia.org/wiki/Najmeh_Abtin'},
  {'title': 'Abu Hafs al-Hashimi al-Qurashi',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hafs_al-Hashimi_al-Qurashi'},
  {'title': 'Abu Haider',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Haider'},
  {'title': 'Eva Abu Halaweh',
   'url': 'https://simple.wikipedia.org/wiki/Eva_Abu_Halaweh'},
  {'title': 'Abu Hamza al-Masri',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hamza_al-Masri'},
  {'title': 'Abu Khaled',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Khal

In [ ]:


# # ---- Framework function ----
# def scrape_wikipedia(urls: Union[str, List[str]], cat_workers: int = 5) -> List[dict]:
#     """
#     General Wikipedia scraper framework.
    
#     - Accepts single URL or list of URLs (pages or categories)
#     - Automatically detects:
#         - Simple vs normal Wikipedia
#         - Category vs single page
#     - Expands categories to individual page URLs
#     - Scrapes pages in parallel using ThreadPoolExecutor
#     - Directs to the appropriate scraper function
#     """

#     if isinstance(urls, str):
#         urls = [urls]

#     # Step 1: Expand all category URLs
#     expanded_urls = []
#     for url in urls:
#         if "/wiki/Category:" in url:
#             categories = get_category_pages(url)
#             pages_urls = [page.get('url', None) for page in categories['pages']]
#             expanded_urls.extend(pages_urls)
#         else:
#             expanded_urls.append(url)

#     # Step 2: Determine which scraper to use for each URL
#     def scrape_dispatcher(url: str):
#         if "simple.wikipedia.org" in url:
#             return scrape_simple_wiki(url)
#         else:
#             return scrape_normal_wiki(url)

#     # Step 3: Parallel scraping
#     results = []
#     with ThreadPoolExecutor(max_workers=cat_workers) as executor:
#         futures = [executor.submit(scrape_dispatcher, u) for u in expanded_urls]
#         for f in futures:
#             results.append(f.result())

#     return results

In [ ]:
# # --------- Check if page is already in DB ----------------------------
# # ------------- or should be scraped ----------------------------------



# def page_needs_scraping(url: str, cur) -> bool:
#     """
#     Returns True if the page (simple or technical) is NOT yet stored.
#     Infers everything from the URL.
#     """

#     # Extract title
#     path = urlparse(url).path
#     if "/wiki/" not in path:
#         return False  # Not a valid wiki page

#     title = unquote(path.split("/wiki/")[-1])

#     # Determine which indicator column to check
#     if "simple.wikipedia.org" in url:
#         indicator_col = "has_simple"
#     elif "wikipedia.org" in url:
#         indicator_col = "has_technical"
#     else:
#         return False  # Not supported domain

#     # 3. Query DB
#     cur.execute(f"SELECT {indicator_col} FROM pages WHERE title = ?", (title,))
#     row = cur.fetchone()

#     if row is None:
#         # Page not in DB at all → needs scraping
#         return True

#     # If indicator is 0 → needs scraping
#     return row[0] == 0



In [ ]:
# --------- Check if page is already in DB ----------------------------
# ------------- or should be scraped ----------------------------------



def page_needs_scraping(url: str, db_path: str) -> bool:
    """
    Returns True if the page (simple or technical) is NOT yet stored.
    Infers everything from the URL.
    """

    # Extract title
    path = urlparse(url).path
    if "/wiki/" not in path:
        return False  # Not a valid wiki page

    title = unquote(path.split("/wiki/")[-1])

    # Determine which indicator column to check
    if "simple.wikipedia.org" in url:
        indicator_col = "has_simple"
    elif "wikipedia.org" in url:
        indicator_col = "has_technical"
    else:
        return False  # Not supported domain

    # Open connection (thread-safe pattern)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    try:
        cur.execute(
            f"SELECT {indicator_col} FROM pages WHERE title = ?",
            (title,)
        )
        row = cur.fetchone()
    finally:
        conn.close()

    if row is None:
        # Page not in DB at all → needs scraping
        return True

    # If indicator is 0 → needs scraping
    return row[0] == 0

In [ ]:

page_needs_scraping(url='https://simple.wikipedia.org/wiki/Markov_chain', db_path="WikipediaOne.db")
page_needs_scraping(url='https://simple.wikipedia.org/wiki/Stochastic_process', db_path="WikipediaOne.db")

True

In [ ]:


def store_page(article_data: dict, conn, cur):
    """
    Stores a scraped Wikipedia article into DB.
    
    article_data = {
        "url": str,
        "title": str,
        "sections": list,
        ...
    }
    """

    url = article_data["url"]
    title = article_data["title"]
    sections = article_data.get("sections", [])
    content = json.dumps(sections)  # store sections as JSON string

    # Determine kind from URL
    kind = "simple" if "simple.wikipedia.org" in url else "technical"

    source = url
    created_at = datetime.now(timezone.utc).isoformat()

    # Ensure page exists
    cur.execute(
        "INSERT OR IGNORE INTO pages (title) VALUES (?);",
        (title,)
    )

    # Get page_id
    cur.execute("SELECT id FROM pages WHERE title = ?", (title,))
    page_id = cur.fetchone()[0]

    # Insert definition or update existing
    cur.execute(
        """
        INSERT OR REPLACE INTO definitions
        (page_id, kind, content, source, created_at)
        VALUES (?, ?, ?, ?, ?);
        """,
        (page_id, kind, content, source, created_at)
    )

    # Update indicator & commit
    cur.execute(f"UPDATE pages SET has_{kind} = 1 WHERE id = ?", (page_id,))

    conn.commit()


In [ ]:


# ---- Framework function ----
def scrape_wikipedia(
        urls: Union[str, List[str]], 
        db_path: str, 
        cat_workers: int = 5
    ) -> List[dict]:
    """
    General Wikipedia scraper framework.
    
    - Accepts single URL or list of URLs (pages or categories)
    - Automatically detects:
        - Simple vs normal Wikipedia
        - Category vs single page
    - Expands categories to individual page URLs
    - Scrapes pages in parallel using ThreadPoolExecutor
    - Directs to the appropriate scraper function
    """

    if isinstance(urls, str):
        urls = [urls]

    # Step 1: Expand all category URLs
    expanded_urls = []
    for url in urls:
        if "/wiki/Category:" in url:
            categories = get_category_pages(url)
            pages_urls = [page.get('url', None) for page in categories['pages']]
            expanded_urls.extend(pages_urls)
        else:
            expanded_urls.append(url)
    
    # Decide which need scraping (don't scrape if already in db)     
    urls_to_scrape = []
    for url in expanded_urls:
        if page_needs_scraping(url, db_path=db_path):
            urls_to_scrape.append(url)
            
    if not urls_to_scrape:
        print("All pages already in DB. Nothing to scrape.")
        return []
    

    # Step 2: Determine which scraper to use for each URL
    def scrape_dispatcher(url: str):
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        
        if "simple.wikipedia.org" in url:
            result = scrape_simple_wiki(url)
        else:
            result = scrape_normal_wiki(url)
        store_page(result, conn, cur)
        conn.close()
        
        return result

    # Step 3: Parallel scraping
    results = []
    with ThreadPoolExecutor(max_workers=cat_workers) as executor:
        futures = [executor.submit(scrape_dispatcher, u) for u in urls_to_scrape]
        for f in futures:
            results.append(f.result())

    return results

In [35]:
pwd

'/home/schmi/projects/explain2me/data'

In [39]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Markov_chain', db_path='WikipediaOne.db')

[{'url': 'https://simple.wikipedia.org/wiki/Markov_chain',
  'title': 'Markov chain',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['A Markov chain is a model of some random process that happens over time. Markov chains are called that because they follow a rule called the Markov property. The Markov property says that whatever happens next in a process only depends on how it is right now (the state). It doesn\'t have a "memory" of how it was before. It is helpful to think of a Markov chain as evolving through discrete steps in time, although the "step" doesn\'t need to have anything to do with time.',
     "Markov chains can be discrete or continuous. Discrete Time Markov Chains are split up into discrete time steps, like t = 1, t = 2, t = 3, and so on. The probability that a chain will go from one state to another state depends only on the state that it's in right now. Continuous Time Markov Chains are chains where the time spent in each state is a real number. The am

In [40]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Stochastic_process', db_path='WikipediaOne.db')

[{'url': 'https://simple.wikipedia.org/wiki/Stochastic_process',
  'title': 'Stochastic process',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['A Stochastic process is a mathematical description of random events that occur one after another. It is possible to order these events according to the time at which they occur.',
     "This can be used to model such things as daily weather data, or exchange rate changes, or medical information like a patient's EKG, EEG, blood pressure or temperature.",
     'Stochastic processes used in various disciplines, including physics, biology, finance, telecommunications, and operations research. They provide a powerful framework for analyzing and predicting the behavior of systems under uncertain conditions.']}],
  'categories': ['Statistics'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Statistics']}]

In [43]:
test1 = ['https://simple.wikipedia.org/wiki/Stochastic_process',
         'https://simple.wikipedia.org/wiki/Category:Statistics',]

test1_res = scrape_wikipedia(urls=test1, db_path='WikipediaOne.db')
len(test1_res), test1_res

(82,
 [{'url': 'https://simple.wikipedia.org/wiki/Stochastic_process',
   'title': 'Stochastic process',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['A Stochastic process is a mathematical description of random events that occur one after another. It is possible to order these events according to the time at which they occur.',
      "This can be used to model such things as daily weather data, or exchange rate changes, or medical information like a patient's EKG, EEG, blood pressure or temperature.",
      'Stochastic processes used in various disciplines, including physics, biology, finance, telecommunications, and operations research. They provide a powerful framework for analyzing and predicting the behavior of systems under uncertain conditions.']}],
   'categories': ['Statistics'],
   'category_urls': ['https://simple.wikipedia.org/wiki/Category:Statistics']},
  {'url': 'https://simple.wikipedia.org/wiki/Regression_toward_the_mean',
   'title': 'Regression tow